# 🫀 PTB-XL Notebook 12 — Attribution Overlays for Confident and Unconfident Elderly Cases

## Executive Summary & Purpose

Review Item 7 asked for literal ECG-with-heat-map examples alongside the quantitative
importance-score comparison the deck already had. A reader should be able to *see* what an
attribution map looks like when the model is sure about an 80+ patient and when it is not.

For CD and HYP — the two superclasses the model discriminates worst — this notebook picks the
most- and least-confident **correctly labelled** 80+ TEST record by predictive entropy and draws,
side by side, the Integrated-Gradients overlay on all 12 leads plus a Grad-CAM-1D temporal strip.
The attribution code is the same as NB06's.

The point of the figure is that the attribution concentrates on the same morphological region in
both columns. That is the visual counterpart of the ρ = 0.955 vs 0.980 rank agreement reported
in NB06: a weak confidence effect, not the collapse an earlier draft claimed.

**Outputs** — `outputs/figures/nb12_xai_pairs/12_xai_pair_{CD,HYP}.png` and
`outputs/dataset_validation/12_xai_pair_cases.json` recording exactly which records were chosen.

## Module 0 — Imports, Seeds and Paths

Imports, the global seed, and the project paths. The seed is set here and nowhere else, so every stochastic step below inherits it.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
# notebooks have no __file__; resolve the project root from the working directory
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed'
OUT = ROOT / 'outputs' / 'figures' / 'nb12_xai_pairs'
OUT.mkdir(parents=True, exist_ok=True)

SUPER = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
SIG_LEN, FS, IG_STEPS = 1000, 100.0, 64
DEVICE = torch.device('cpu')

## Module 1 — The Attribution Model

InceptionTime1D, the signal-only anchor architecture, defined verbatim as in NB05/NB06. Attribution is computed on this single model rather than the ensemble: Integrated Gradients needs one differentiable path from input to logit, and the ensemble's AUC-weighted average of three architectures does not give a single coherent one.

In [ ]:
class InceptionBlock1D(nn.Module):
    def __init__(self, in_channels, nb_filters=32, kernel_sizes=(9, 19, 39)):
        super().__init__()
        self.bottleneck = nn.Conv1d(in_channels, nb_filters, 1, bias=False)
        self.convs = nn.ModuleList([nn.Conv1d(nb_filters, nb_filters, k, padding=k // 2, bias=False)
                                    for k in kernel_sizes])
        self.pool_conv = nn.Conv1d(in_channels, nb_filters, 1, bias=False)
        self.pool = nn.MaxPool1d(3, stride=1, padding=1)
        self.bn = nn.BatchNorm1d(nb_filters * (len(kernel_sizes) + 1))
        self.act = nn.ReLU()
    def forward(self, x):
        b = self.bottleneck(x)
        return self.act(self.bn(torch.cat([c(b) for c in self.convs] + [self.pool_conv(self.pool(x))], 1)))

class InceptionTime1D(nn.Module):
    def __init__(self, num_classes=5, in_channels=12, nb_filters=32, depth=4):
        super().__init__()
        ch = [in_channels] + [nb_filters * 4] * depth
        self.blocks = nn.Sequential(*[InceptionBlock1D(ch[i], nb_filters) for i in range(depth)])
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(nb_filters * 4, num_classes)
        self.drop = nn.Dropout(0.3)
    def forward(self, x):
        return self.fc(self.drop(self.gap(self.blocks(x)).squeeze(-1)))

## Module 2 — Data, Checkpoint and Predictive Entropy

Loads the TEST split and the released checkpoint, scores every record, and computes mean binary entropy across the five classes as the confidence measure. Entropy rather than the predicted probability of the target class, because we want records where the model is globally decisive or globally hesitant, not just confident about one label.

In [ ]:
X_test = np.transpose(np.load(DATA / 'X_test_clean_100.npy'), (0, 2, 1)).astype(np.float32)
y_test = np.load(DATA / 'y_test_diag_100.npy').astype(np.float32)
age_test = np.load(DATA / 'y_test_age_100.npy').astype(np.float32)

model = InceptionTime1D(5)
model.load_state_dict(torch.load(ROOT / 'outputs' / '05_optimized_InceptionTime1D_best.pth',
                                 map_location=DEVICE, weights_only=True))
model.eval().to(DEVICE)

@torch.no_grad()
def predict(X, bs=128):
    z = np.vstack([model(torch.from_numpy(X[i:i + bs])).numpy() for i in range(0, len(X), bs)])
    return 1.0 / (1.0 + np.exp(-z))

P = predict(X_test)
pe = np.clip(P, 1e-7, 1 - 1e-7)
entropy = (-(pe * np.log(pe) + (1 - pe) * np.log(1 - pe))).mean(1)

## Module 3 — Integrated Gradients and Grad-CAM-1D

Integrated Gradients with 64 steps and a zero baseline gives per-sample, per-lead attribution. Grad-CAM-1D on the fourth Inception block gives a coarse temporal saliency strip. Showing both guards against reading too much into either: IG is fine-grained but noisy, Grad-CAM is smooth but low-resolution.

In [ ]:
def integrated_gradients(x_np, c, steps=IG_STEPS):
    xt = torch.tensor(x_np[None]); bt = torch.zeros_like(xt)
    a = torch.linspace(0, 1, steps).view(-1, 1, 1)
    path = (bt + a * (xt - bt)).requires_grad_(True)
    g = torch.autograd.grad(model(path)[:, c].sum(), path)[0].mean(0)
    return ((xt - bt)[0] * g).detach().numpy()

def grad_cam_1d(x_np, c, block_idx=3):
    acts, grads = {}, {}
    layer = model.blocks[block_idx]
    h1 = layer.register_forward_hook(lambda m, i, o: acts.__setitem__('a', o))
    h2 = layer.register_full_backward_hook(lambda m, gi, go: grads.__setitem__('g', go[0]))
    x = torch.tensor(x_np[None], requires_grad=True)
    model.zero_grad(); model(x)[0, c].backward()
    h1.remove(); h2.remove()
    cam = torch.relu((grads['g'][0].mean(1)[:, None] * acts['a'][0]).sum(0)).detach().numpy()
    return cam / (cam.max() + 1e-9)

## Module 4 — Case Selection and the Paired Overlay Figures

Within the 80+ cohort, for each of CD and HYP, the correctly-labelled true positives are ranked by entropy and the extremes are taken. Both panels are therefore cases the model got *right* — the comparison is about confidence, not about correctness, which is what makes the stability of the attribution meaningful.

In [ ]:
elderly = age_test >= 80
picked = {}
t = np.arange(SIG_LEN) / FS
for k, sc in enumerate(SUPER):
    if sc not in ('CD', 'HYP'):
        continue
    pos = np.where(elderly & (y_test[:, k] == 1) & (P[:, k] >= 0.5))[0]
    if len(pos) < 2:
        print(f'[skip] {sc}: only {len(pos)} confident 80+ true positives'); continue
    hi = pos[np.argmin(entropy[pos])]       # model most certain overall
    lo = pos[np.argmax(entropy[pos])]       # model least certain overall
    picked[sc] = dict(high=int(hi), low=int(lo),
                      p_high=float(P[hi, k]), p_low=float(P[lo, k]),
                      entropy_high=float(entropy[hi]), entropy_low=float(entropy[lo]),
                      age_high=float(age_test[hi]), age_low=float(age_test[lo]),
                      labels_high=[SUPER[i] for i in np.where(y_test[hi] == 1)[0]],
                      labels_low=[SUPER[i] for i in np.where(y_test[lo] == 1)[0]])

    fig, axes = plt.subplots(13, 2, figsize=(13.5, 14), sharex=True,
                             gridspec_kw={'height_ratios': [1] * 12 + [0.55]})
    for col, (tag, j) in enumerate([('high-confidence', hi), ('low-confidence', lo)]):
        ig = integrated_gradients(X_test[j], k)
        cam = grad_cam_1d(X_test[j], k)
        a = np.abs(ig); a = a / (a.max() + 1e-9)
        for l in range(12):
            ax = axes[l, col]
            ax.plot(t, X_test[j][l], color='0.15', lw=0.6)
            ax.scatter(t, X_test[j][l], c=a[l], cmap='Reds', s=4, vmin=0, vmax=1)
            ax.set_yticks([])
            if col == 0:
                ax.set_ylabel(LEADS[l], rotation=0, ha='right', va='center', fontsize=8)
        axes[12, col].imshow(cam[None], aspect='auto', cmap='viridis', extent=[t[0], t[-1], 0, 1])
        axes[12, col].set_yticks([]); axes[12, col].set_xlabel('time (s)')
        if col == 0:
            axes[12, col].set_ylabel('Grad-CAM', rotation=0, ha='right', va='center', fontsize=8)
        axes[0, col].set_title(f'{tag}: TEST idx {j}, age {age_test[j]:.0f}, '
                               f'p({sc})={P[j, k]:.2f}, entropy {entropy[j]:.2f}\n'
                               f'true labels: {", ".join(SUPER[i] for i in np.where(y_test[j] == 1)[0])}',
                               fontsize=9)
    fig.suptitle(f'{sc}: Integrated-Gradients overlay and Grad-CAM-1D for a confident and an '
                 f'unconfident correctly-labelled 80+ patient', fontsize=12)
    fig.tight_layout()
    fig.savefig(OUT / f'12_xai_pair_{sc}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'[saved] {sc}: high idx {hi} (p={P[hi,k]:.2f}) vs low idx {lo} (p={P[lo,k]:.2f})')

## Module 5 — Record the Chosen Cases

The selected record indices, their probabilities, entropies, ages and true labels are written to JSON. Without this the figures would not be traceable to specific records, which is the standard this project now holds itself to.

In [ ]:
with open(ROOT / 'outputs' / 'dataset_validation' / '12_xai_pair_cases.json', 'w') as f:
    json.dump(picked, f, indent=1)
print(json.dumps(picked, indent=1))

## Module 6 — Figures Produced by This Notebook

The plotting code above keeps the script's non-interactive backend and writes each figure to
disk, so this cell displays the saved PNGs inline. The files on disk are the ones the deck and
the manuscript reference.

In [ ]:
from pathlib import Path
from IPython.display import Image, display, Markdown

for folder in ['outputs/figures/nb12_xai_pairs']:
    for png in sorted((ROOT / folder).glob('*.png')):
        display(Markdown(f'**{png.name}**'))
        display(Image(filename=str(png)))